# 🔐 Sentinel Cyber AI — Google Colab Training

Fine-tune an open-source LLM for **cybersecurity vulnerability detection** using QLoRA.

**Runtime:** Go to `Runtime → Change runtime type → Select T4 GPU` for best performance.

**Time:** ~30-60 minutes on a free T4 GPU.

In [ ]:
# @title 1. Install Dependencies

# Install unsloth (optimized training)
!pip install "unsloth[cu124] @ git+https://github.com/unslothai/unsloth.git"

# Install other dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes pyyaml

In [ ]:
# @title 2. Mount Google Drive (for saving models)
from google.colab import drive
drive.mount('/content/drive')

print('✅ Google Drive mounted!')
print('Models will be saved to: /content/drive/MyDrive/sentinel-cyber-ai/')

In [ ]:
# @title 3. Check Hardware
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: No GPU detected! Training will be very slow.")

In [ ]:
# @title 4. Create the Cybersecurity Dataset
import json
import os

# Download the project files to Colab
!wget -q https://raw.githubusercontent.com/your-repo/sentinel-cyber-ai/main/scripts/prepare_dataset.py -O prepare_dataset.py || echo "Will use built-in dataset instead"

# Use the built-in cybersecurity examples
VULNERABLE_CODE_EXAMPLES = [
    {
        "instruction": "Analyze this code for security vulnerabilities and provide a fix.",
        "input": """def get_user(username):\n    query = f"SELECT * FROM users WHERE username = '{username}'"\n    cursor.execute(query)\n    return cursor.fetchall()""",
        "output": "## Vulnerability: SQL Injection\n\n**Severity: CRITICAL**\n\nThe code directly interpolates user input into a SQL query string, allowing an attacker to manipulate the query by providing crafted input like `' OR '1'='1`.\n\n### Fix:\n```python\ndef get_user(username):\n    query = \"SELECT * FROM users WHERE username = ?\"\n    cursor.execute(query, (username,))\n    return cursor.fetchall()\n```\n\nAlways use parameterized queries or an ORM to safely handle user input in database operations."
    },
    {
        "instruction": "Is this code vulnerable? If so, explain the vulnerability and fix it.",
        "input": """function displayMessage() {\n    const name = new URLSearchParams(window.location.search).get('name');\n    document.getElementById('greeting').innerHTML = 'Hello, ' + name + '!';\n}""",
        "output": "## Vulnerability: Cross-Site Scripting (XSS)\n\n**Severity: HIGH**\n\nThe code inserts unsanitized URL parameter content directly into the DOM using `innerHTML`.\n\n### Fix:\n```javascript\nfunction displayMessage() {\n    const name = new URLSearchParams(window.location.search).get('name');\n    document.getElementById('greeting').textContent = 'Hello, ' + name + '!';\n}\n```\n\nUse `textContent` instead of `innerHTML`."
    },
    {
        "instruction": "Review this code for security flaws and suggest fixes.",
        "input": """import subprocess\n\ndef ping_host(hostname):\n    result = subprocess.run(f"ping -c 4 {hostname}", shell=True, capture_output=True)\n    return result.stdout""",
        "output": "## Vulnerability: Command Injection\n\n**Severity: CRITICAL**\n\nThe code passes user-controlled input directly to a shell command using `shell=True`. An attacker can provide input like `; rm -rf /` to execute arbitrary commands.\n\n### Fix:\n```python\nimport subprocess\n\ndef ping_host(hostname):\n    result = subprocess.run(\n        [\"ping\", \"-c\", \"4\", hostname],\n        capture_output=True,\n        timeout=30\n    )\n    return result.stdout\n```\n\nNever use `shell=True` with user input."
    },
    {
        "instruction": "Find the security vulnerability in this file access function.",
        "input": """def read_user_file(filename):\n    base_path = "/var/data/users/"\n    full_path = base_path + filename\n    with open(full_path, 'r') as f:\n        return f.read()""",
        "output": "## Vulnerability: Path Traversal\n\n**Severity: HIGH**\n\nThe `filename` parameter is concatenated directly without sanitization. An attacker can use `../` sequences to read arbitrary files.\n\n### Fix:\n```python\nimport os\n\ndef read_user_file(filename):\n    base_path = "/var/data/users/"\n    full_path = os.path.normpath(os.path.join(base_path, filename))\n    if not full_path.startswith(os.path.normpath(base_path)):\n        raise SecurityError(\"Path traversal detected\")\n    with open(full_path, 'r') as f:\n        return f.read()\n```"
    },
    {
        "instruction": "Review this code for security vulnerabilities and provide a fix.",
        "input": """import pickle\nimport base64\n\ndef load_session(session_data):\n    return pickle.loads(base64.b64decode(session_data))""",
        "output": "## Vulnerability: Insecure Deserialization\n\n**Severity: CRITICAL**\n\nPython's `pickle` module can execute arbitrary code during deserialization.\n\n### Fix:\n```python\nimport json\nimport base64\n\ndef load_session(session_data):\n    return json.loads(base64.b64decode(session_data).decode('utf-8'))\n```\n\nNever use `pickle` with untrusted data."
    },
    {
        "instruction": "Analyze this form handler for security vulnerabilities.",
        "input": """@app.route('/transfer', methods=['POST'])\ndef transfer_money():\n    to_account = request.form['to_account']\n    amount = request.form['amount']\n    current_user.transfer(to_account, amount)\n    return redirect('/success')""",
        "output": "## Vulnerability: Cross-Site Request Forgery (CSRF)\n\n**Severity: HIGH**\n\nThis endpoint performs a sensitive action without verifying the request origin. An attacker can create a malicious page that auto-submits to this endpoint.\n\n### Fix:\n```python\nfrom flask_wtf.csrf import CSRFProtect\ncsrf = CSRFProtect(app)\n\n@app.route('/transfer', methods=['POST'])\ndef transfer_money():\n    to_account = request.form['to_account']\n    amount = request.form['amount']\n    # Re-authenticate for sensitive actions\n    if not request.form.get('confirm_password'):\n        return redirect('/transfer?error=reauth_required')\n    current_user.transfer(to_account, amount)\n    return redirect('/success')\n```\n\nUse CSRF tokens on all state-changing forms."
    },
    {
        "instruction": "Find the cryptographic weakness in this code.",
        "input": """import random\n\ndef generate_reset_token():\n    chars = 'abcdefghijklmnopqrstuvwxyz0123456789'\n    token = ''.join(random.choice(chars) for _ in range(20))\n    return token""",
        "output": "## Vulnerability: Insecure Randomness\n\n**Severity: MEDIUM**\n\nThe `random` module uses Mersenne Twister PRNG, which is predictable.\n\n### Fix:\n```python\nimport secrets\n\ndef generate_reset_token():\n    return secrets.token_urlsafe(32)\n```\n\nUse `secrets` module for security-related randomness."
    },
    {
        "instruction": "Explain CWE-89 and how to prevent it.",
        "input": "CWE-89: SQL Injection",
        "output": "## CWE-89: SQL Injection\n\n**Description:** The product constructs SQL commands using externally-influenced input without properly neutralizing special elements.\n\n**Impact:** Attackers can read, modify, or delete database contents; execute administrative operations.\n\n**Prevention:**\n1. Use parameterized queries / prepared statements\n2. Use ORM frameworks with safe query builders\n3. Implement least privilege accounts\n4. Validate and sanitize all inputs\n5. Use stored procedures with parameterized calls"
    },
    {
        "instruction": "Explain CWE-79 and how to prevent it.",
        "input": "CWE-79: Cross-Site Scripting",
        "output": "## CWE-79: Cross-Site Scripting (XSS)\n\n**Description:** The product does not neutralize user-controllable input before it is placed in output used as a web page.\n\n**Variants:**\n- **Reflected XSS:** Script comes from the current HTTP request\n- **Stored XSS:** Script is stored on the server\n- **DOM-based XSS:** Vulnerability in client-side JavaScript\n\n**Prevention:**\n1. Use context-aware output encoding\n2. Implement Content Security Policy (CSP)\n3. Use frameworks that auto-escape\n4. Validate and sanitize input\n5. Use `textContent` instead of `innerHTML`"
    },
    {
        "instruction": "How should passwords be stored securely?",
        "input": "I need to store user passwords in my database.",
        "output": "## Secure Password Storage\n\n### Correct approach:\n```python\nimport bcrypt\n\ndef hash_password(password: str) -> str:\n    salt = bcrypt.gensalt(rounds=12)\n    hashed = bcrypt.hashpw(password.encode('utf-8'), salt)\n    return hashed.decode('utf-8')\n\ndef verify_password(password: str, hashed: str) -> bool:\n    return bcrypt.checkpw(password.encode('utf-8'), hashed.encode('utf-8'))\n```\n\n### Key principles:\n1. Use a **slow hash** function (bcrypt, scrypt, argon2)\n2. Always use a **unique salt** per password\n3. Use a **high work factor**\n4. Never roll your own cryptography"
    },
]

# Save dataset
os.makedirs('/content/data/processed', exist_ok=True)
with open('/content/data/processed/cyber_train.jsonl', 'w') as f:
    for ex in VULNERABLE_CODE_EXAMPLES:
        f.write(json.dumps(ex) + '\n')

print(f'✅ Dataset created with {len(VULNERABLE_CODE_EXAMPLES)} examples')

In [ ]:
# @title 5. 🚀 Start Training

# Model: DeepSeek-R1-Distill-Qwen-7B (best reasoning-to-size ratio for Colab free tier)
BASE_MODEL = "unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit"

# Or use a smaller model if you want faster training:
# BASE_MODEL = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit"

print(f"Using model: {BASE_MODEL}")
print("This will take 30-60 minutes on a T4 GPU...")

In [ ]:
# @title 6. Load Model & Tokenizer
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=True,
    device_map="auto",
)

print(f"✅ Model loaded: {model.num_parameters():,} parameters")

In [ ]:
# @title 7. Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("✅ LoRA applied!")

In [ ]:
# @title 8. Prepare Dataset
from datasets import Dataset
import json

# Load the dataset
data = []
with open('/content/data/processed/cyber_train.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

dataset = Dataset.from_list(data)

# Format with chat template
def format_fn(examples):
    texts = []
    for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        user_msg = f"{inst}\n\n{inp}" if inp else inst
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": out},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

dataset = dataset.map(format_fn, batched=True)
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(split['train'])} examples")
print(f"Val: {len(split['test'])} examples")

# Show a formatted example
print("\n📝 Example formatted training sample:")
print(split['train'][0]['text'][:500] + "...")

In [ ]:
# @title 9. Train!
from trl import SFTTrainer
from transformers import TrainingArguments

output_dir = "/content/drive/MyDrive/sentinel-cyber-ai"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    remove_unused_columns=False,
    save_total_limit=2,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    args=training_args,
    max_seq_length=max_seq_length,
    dataset_text_field="text",
)

print("🚀 Training started!")
print("This will take ~30-60 minutes...")
trainer.train()

print("✅ Training complete!")

In [ ]:
# @title 10. Save the Fine-Tuned Model
import os

# Save LoRA adapters
adapter_path = os.path.join(output_dir, "sentinel-v1-adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"✅ LoRA adapters saved to: {adapter_path}")

# Optional: Save merged model (full weights, no separate adapter)
try:
    merged_path = os.path.join(output_dir, "sentinel-v1-merged")
    merged_model = FastLanguageModel.get_merged_model(model)
    merged_model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)
    print(f"✅ Merged model saved to: {merged_path}")
except Exception as e:
    print(f"Note: Could not save merged model: {e}")
    print("The LoRA adapter is sufficient for inference.")

In [ ]:
# @title 11. Test Your Model!
FastLanguageModel.for_inference(model)

test_prompts = [
    "Analyze this code for security vulnerabilities:\n\nprint(eval(input('Enter expression: ')))",
    "Find flaws in:\n\n@app.route('/search')\ndef search():\n    return db.execute(f\"SELECT * FROM items WHERE q = '{request.args[\"q\"]}'\")",
    "How do I prevent XSS in my React app?",
]

for prompt in test_prompts:
    print("\n" + "═" * 60)
    print(f"📝 Prompt: {prompt[:80]}..." if len(prompt) > 80 else f"📝 Prompt: {prompt}")
    print("═" * 60)
    
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.95,
        repetition_penalty=1.1,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"🤖 Response:\n{response}\n")

---
## 🎉 You've trained Sentinel Cyber AI!

### Next steps:
1. **Download your model** from Google Drive
2. **Run inference locally** using `scripts/inference.py`
3. **Add more training data** — curate your own vulnerable→secure code pairs
4. **Build a security agent** wrapping the model with tools like CodeQL

### Need to start over?
Go to `Runtime → Factory reset runtime` to clear everything.